# Kahneman Loss/Gain Framing RCT with TRIBE v2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Kahneman_Framing_RCT.ipynb)

Text-only in-silico RCT: hundreds of matched gain/loss framing texts through [facebook/tribev2](https://huggingface.co/facebook/tribev2).

**Setup**
1. Runtime → **A100 GPU** (40 GB+ VRAM)
2. Colab secret `HF_TOKEN` = Hugging Face read token with [LLaMA 3.2 access](https://huggingface.co/meta-llama/Llama-3.2-3B)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!pip install -q 'tribev2[plotting] @ git+https://github.com/facebookresearch/TRIBEv2.git' scipy pyyaml

In [ ]:
import os, sys
from pathlib import Path

REPO = Path('/content/DSprojects')
if not REPO.exists():
    !git clone -q https://github.com/akifnu/DSprojects.git /content/DSprojects

ROOT = REPO / 'tribev2'
sys.path.insert(0, str(ROOT / 'src'))
%cd {ROOT}

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['HF_HUB_HTTP_TIMEOUT'] = '600'

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('Ready:', ROOT)

In [ ]:
!PYTHONPATH=src python scripts/generate_rct_dataset.py

import json
from pathlib import Path
meta = json.loads(Path('data/framing_rct/protocol.json').read_text())
print(f"Scenarios: {meta['n_scenarios']}")
print(f"Unique texts: {meta['n_scenarios'] * 2}")
print(f"Subjects: {meta['n_subjects']}")
print(f"Trials: {meta['n_trials']}")

In [ ]:
# Start with 12 scenario pairs on Colab; raise for full study (318 pairs)
!PYTHONPATH=src python scripts/run_framing_rct.py --preload-llama --device cuda --max-scenarios 12

In [ ]:
import json
from pathlib import Path
from IPython.display import JSON

report = json.loads(Path('outputs/reports/framing_rct_analysis.json').read_text())
print(report['kahneman_alignment']['interpretation'])
JSON(report)